# Notebook 04: Analysis & Paper Tables

Dravidian Extension — Telugu / Tamil / Kannada, generic vs. localized conditions.

Produces all statistics, tables, and figures for the paper.

**Inputs:**
- `../results/checkpoints/raw_responses.json` (from `01_run_inference.py`) — used for script-fidelity, since `judge_scores.json` only contains records that already passed the fidelity filter
- `../data/judge_scores.json` (from `02_judge_responses.py`)
- `../data/embed_distances.json` (from `03_embed_distance.py`)

**Outputs:** `../results/final_table.csv`, figures in `../results/figures/`

**Note:** this pipeline has no `religion` field and no Hindi/Punjabi — those belong to the *reference* repo this project was adapted from. Here the two axes are `lang` (`en`/`te`/`ta`/`kn`) and `version` (`generic`/`localized`).

**On what follows:** every number below is computed directly from the three JSON files your pipeline produces — no simulated, estimated, or placeholder values anywhere. Sections beyond the plain mean tables that `02`/`03` already print (i.e. the significance tests, correlations, and rankings) are an added analysis layer built on top of that real data; they are not themselves part of `01`–`03`'s own logic, and are called out as such at each step.

In [ ]:
!pip install -q scipy pandas matplotlib seaborn

In [ ]:
import json, os
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

BASE     = Path(".")
DATA_DIR = BASE / "../data"
CKPT_DIR = BASE / "../results/checkpoints"
RES_DIR  = BASE / "../results"
FIG_DIR  = RES_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

raw_path   = CKPT_DIR / "raw_responses.json"
judge_path = DATA_DIR / "judge_scores.json"
embed_path = DATA_DIR / "embed_distances.json"

for p in (raw_path, judge_path, embed_path):
    if not p.exists():
        raise FileNotFoundError(
            f"{p} not found. Run 01_run_inference.py, 02_judge_responses.py, "
            f"and 03_embed_distance.py first."
        )

with open(raw_path)   as f: raw_data   = json.load(f)
with open(judge_path) as f: judge_data = json.load(f)
with open(embed_path) as f: embed_data = json.load(f)

rdf = pd.DataFrame(raw_data)      # model, scenario_id, dimension, version, lang, region, response, script_ok, truncated
jdf = pd.DataFrame(judge_data)    # model, scenario_id, dimension, version, lang, script_ok, truncated, stance, reasoning, response_snippet
edf = pd.DataFrame(embed_data)    # model, scenario_id, dimension, version, lang_pair, anchor, cosine_sim, drift

LANGS           = ["en", "te", "ta", "kn"]
DRAVIDIAN_LANGS = ["te", "ta", "kn"]
VERSIONS        = ["generic", "localized"]

# Valid judge scores only (stance == 0 means the judge saw an empty/off-topic/
# refused response; stance == -1 means the judge call itself failed).
jdf = jdf[jdf["stance"] > 0].copy()

print(f"Raw records:        {len(rdf)}")
print(f"Judge records (valid stance): {len(jdf)}")
print(f"Embed records:      {len(edf)}")
print(f"Models: {sorted(rdf['model'].unique())}")
print(f"Dimensions: {sorted(rdf['dimension'].unique())}")

## Table 1 — Script Fidelity
Computed from `raw_responses.json` (see note in the intro cell — `judge_scores.json` would read 100% by construction).

In [ ]:
fidelity_rows = []
for model in rdf["model"].unique():
    for lang in DRAVIDIAN_LANGS:
        sub = rdf[(rdf["model"] == model) & (rdf["lang"] == lang)]
        pass_rate = sub["script_ok"].mean() * 100 if len(sub) else float("nan")
        fidelity_rows.append({"Model": model, "Language": lang, "Script Fidelity %": round(pass_rate, 1)})

fidelity_df = pd.DataFrame(fidelity_rows).pivot(index="Model", columns="Language", values="Script Fidelity %")
fidelity_df = fidelity_df[[c for c in DRAVIDIAN_LANGS if c in fidelity_df.columns]]
print("TABLE 1: Script Fidelity")
print(fidelity_df.to_string())

## Table 2 — Cross-lingual Value Drift (judge scores, generic condition)
Restricted to `version=="generic"` — see the intro cell's note on why the localized EN anchor can't be paired the same way.

In [ ]:
jdf_generic = jdf[jdf["version"] == "generic"]

drift_rows = []
models = jdf_generic["model"].unique()
dims   = jdf_generic["dimension"].unique()

for model in models:
    for dim in dims:
        sub = jdf_generic[(jdf_generic["model"] == model) & (jdf_generic["dimension"] == dim)]
        mean_en = sub[sub["lang"] == "en"]["stance"].mean()
        mean_te = sub[sub["lang"] == "te"]["stance"].mean()
        mean_ta = sub[sub["lang"] == "ta"]["stance"].mean()
        mean_kn = sub[sub["lang"] == "kn"]["stance"].mean()
        drift_rows.append({
            "model":       model,
            "dimension":   dim,
            "stance_EN":   round(mean_en, 2),
            "stance_TE":   round(mean_te, 2),
            "stance_TA":   round(mean_ta, 2),
            "stance_KN":   round(mean_kn, 2),
            "drift_EN_TE": round(abs(mean_en - mean_te), 2),
            "drift_EN_TA": round(abs(mean_en - mean_ta), 2),
            "drift_EN_KN": round(abs(mean_en - mean_kn), 2),
        })

drift_df = pd.DataFrame(drift_rows)
print("TABLE 2: Cross-lingual Value Drift (judge scores, generic condition)")
print(drift_df.to_string(index=False))

## H1 / H2 — Significance tests
*Added analysis layer (not in `01`–`03`).* Exclude models with script fidelity failure, determined empirically from Table 1 (below a threshold in ANY Dravidian language) rather than hardcoded model names from a different project.

In [ ]:
FIDELITY_THRESHOLD = 50.0  # percent — a choice made for this analysis, not specified anywhere in the codebase
SCRIPT_FAIL_MODELS = set(
    fidelity_df[(fidelity_df < FIDELITY_THRESHOLD).any(axis=1)].index
)
SCRIPT_CAPABLE = [m for m in jdf_generic["model"].unique() if m not in SCRIPT_FAIL_MODELS]
jdf_stats = jdf_generic[jdf_generic["model"].isin(SCRIPT_CAPABLE)]
print(f"Models excluded from H1/H2 (script failure, <{FIDELITY_THRESHOLD}% in some language): {SCRIPT_FAIL_MODELS or 'none'}")
print(f"Models included in H1/H2 ({len(SCRIPT_CAPABLE)}): {list(SCRIPT_CAPABLE)}")

def paired_stances(target_lang, df=jdf_stats):
    en_vals, tgt_vals, keys = [], [], []
    for _, row in df[df["lang"] == "en"].iterrows():
        tgt_row = df[
            (df["model"] == row["model"]) &
            (df["scenario_id"] == row["scenario_id"]) &
            (df["lang"] == target_lang)
        ]
        if len(tgt_row):
            en_vals.append(row["stance"])
            tgt_vals.append(tgt_row.iloc[0]["stance"])
            keys.append((row["model"], row["scenario_id"], row["dimension"]))
    return np.array(en_vals), np.array(tgt_vals), keys

def cohens_d(a, b):
    return (np.mean(a) - np.mean(b)) / np.sqrt((np.std(a)**2 + np.std(b)**2) / 2)

print()
print("── H1: Is drift(EN→Dravidian) significantly > 0 across script-capable models? ──")
drift_by_lang = {}
pair_keys_by_lang = {}
for lang in DRAVIDIAN_LANGS:
    en_arr, tgt_arr, keys = paired_stances(lang)
    if len(en_arr) < 2:
        print(f"H1 EN vs {lang.upper()}: not enough paired records, skipping")
        continue
    stat, p = stats.wilcoxon(en_arr, tgt_arr)
    drift = np.abs(en_arr - tgt_arr)
    d = cohens_d(tgt_arr, en_arr)
    drift_by_lang[lang] = drift
    pair_keys_by_lang[lang] = keys
    print(f"H1 EN vs {lang.upper()}: W={stat:.1f}, p={p:.4f}, mean_drift={drift.mean():.2f}, Cohen's d={d:.3f}, n={len(drift)}")

In [ ]:
print("── H2a (omnibus): Kruskal-Wallis across EN→TE / EN→TA / EN→KN drift distributions ──")
common_langs = [l for l in DRAVIDIAN_LANGS if l in drift_by_lang]
if len(common_langs) >= 2:
    samples = [drift_by_lang[l] for l in common_langs]
    stat_kw, p_kw = stats.kruskal(*samples)
    print(f"Kruskal-Wallis across {[l.upper() for l in common_langs]}: H={stat_kw:.2f}, p={p_kw:.4f}")
    if p_kw < 0.05:
        print("  -> significant difference somewhere among the three languages; see pairwise tests below")
    else:
        print("  -> no significant omnibus difference detected among the three languages")
else:
    print("Not enough languages with paired data for an omnibus test")

print()
print("── H2b (pairwise): Does drift(EN→Dravidian) differ by language? ──")
lang_pairs = [(a, b) for i, a in enumerate(DRAVIDIAN_LANGS) for b in DRAVIDIAN_LANGS[i+1:]]
h2_rows = []
for a, b in lang_pairs:
    if a in drift_by_lang and b in drift_by_lang and len(drift_by_lang[a]) == len(drift_by_lang[b]):
        stat_h2, p_h2 = stats.wilcoxon(drift_by_lang[a], drift_by_lang[b])
        d_h2 = cohens_d(drift_by_lang[a], drift_by_lang[b])
        test_used = "wilcoxon (paired)"
    elif a in drift_by_lang and b in drift_by_lang:
        stat_h2, p_h2 = stats.mannwhitneyu(drift_by_lang[a], drift_by_lang[b], alternative="two-sided")
        d_h2 = cohens_d(drift_by_lang[a], drift_by_lang[b])
        test_used = "mann-whitney (unpaired, sample sizes differ)"
    else:
        continue
    h2_rows.append({
        "lang_a": a, "lang_b": b, "test": test_used,
        "mean_drift_a": round(drift_by_lang[a].mean(), 3),
        "mean_drift_b": round(drift_by_lang[b].mean(), 3),
        "stat": round(stat_h2, 2), "p": round(p_h2, 4), "cohens_d": round(d_h2, 3),
    })
h2_df = pd.DataFrame(h2_rows)
print(h2_df.to_string(index=False))

## H3 — Does localizing the scenario (generic → localized) change drift?
This is this repo's own H3, matching the built-in summaries at the end of `02_judge_responses.py` / `03_embed_distance.py` — not an "Aya vs other models" comparison, which belonged to a different project's model roster.

Judge-score version paired on `(model, scenario_id, lang)` across version — safe here since native-language rows never collide across region. Embedding version uses `edf`'s already-anchored `drift` column directly.

In [ ]:
print("── H3 (judge stance): generic vs localized, per model × language ──")
h3_judge_rows = []
for model in jdf["model"].unique():
    for lang in DRAVIDIAN_LANGS:
        gen = jdf[(jdf["model"] == model) & (jdf["lang"] == lang) & (jdf["version"] == "generic")]
        loc = jdf[(jdf["model"] == model) & (jdf["lang"] == lang) & (jdf["version"] == "localized")]
        merged = pd.merge(gen[["scenario_id", "stance"]], loc[["scenario_id", "stance"]],
                           on="scenario_id", suffixes=("_generic", "_localized"))
        if len(merged) >= 2:
            stat, p = stats.wilcoxon(merged["stance_generic"], merged["stance_localized"])
        else:
            stat, p = np.nan, np.nan
        h3_judge_rows.append({
            "model": model, "lang": lang, "n_paired": len(merged),
            "mean_generic": round(merged["stance_generic"].mean(), 2) if len(merged) else np.nan,
            "mean_localized": round(merged["stance_localized"].mean(), 2) if len(merged) else np.nan,
            "wilcoxon_p": round(p, 4) if p == p else np.nan,
        })
h3_judge_df = pd.DataFrame(h3_judge_rows)
print(h3_judge_df.to_string(index=False))

print()
print("── H3 (embedding drift): generic vs localized, per model × language pair ──")
h3_embed_rows = []
for model in edf["model"].unique():
    for lp in [f"en→{l}" for l in DRAVIDIAN_LANGS]:
        gen = edf[(edf["model"] == model) & (edf["lang_pair"] == lp) & (edf["version"] == "generic")]
        loc = edf[(edf["model"] == model) & (edf["lang_pair"] == lp) & (edf["version"] == "localized")]
        merged = pd.merge(gen[["scenario_id", "drift"]], loc[["scenario_id", "drift"]],
                           on="scenario_id", suffixes=("_generic", "_localized"))
        if len(merged) >= 2:
            stat, p = stats.wilcoxon(merged["drift_generic"], merged["drift_localized"])
        else:
            stat, p = np.nan, np.nan
        h3_embed_rows.append({
            "model": model, "lang_pair": lp, "n_paired": len(merged),
            "mean_drift_generic": round(merged["drift_generic"].mean(), 3) if len(merged) else np.nan,
            "mean_drift_localized": round(merged["drift_localized"].mean(), 3) if len(merged) else np.nan,
            "wilcoxon_p": round(p, 4) if p == p else np.nan,
        })
h3_embed_df = pd.DataFrame(h3_embed_rows)
print(h3_embed_df.to_string(index=False))

## H3 (pooled) — Aggregate effect across all models & languages combined
*Added analysis layer.* Pools every paired (generic, localized) observation across models and languages into one test, to ask the single overall question: does localization move stance / drift at all, on average, across the whole benchmark?

In [ ]:
print("── H3 pooled: judge stance, generic vs localized (all models × all Dravidian langs combined) ──")
gen_all = jdf[jdf["version"] == "generic"]
loc_all = jdf[jdf["version"] == "localized"]
pooled = pd.merge(
    gen_all[gen_all["lang"].isin(DRAVIDIAN_LANGS)][["model", "scenario_id", "lang", "stance"]],
    loc_all[loc_all["lang"].isin(DRAVIDIAN_LANGS)][["model", "scenario_id", "lang", "stance"]],
    on=["model", "scenario_id", "lang"], suffixes=("_generic", "_localized")
)
if len(pooled) >= 2:
    stat_p, p_p = stats.wilcoxon(pooled["stance_generic"], pooled["stance_localized"])
    d_p = cohens_d(pooled["stance_localized"], pooled["stance_generic"])
    print(f"n={len(pooled)}  mean_generic={pooled['stance_generic'].mean():.3f}  "
          f"mean_localized={pooled['stance_localized'].mean():.3f}")
    print(f"Wilcoxon: W={stat_p:.1f}, p={p_p:.4f}, Cohen's d={d_p:.3f}")
else:
    print("Not enough pooled paired records for this test")

print()
print("── H3 pooled: embedding drift, generic vs localized (all models × all lang pairs combined) ──")
pooled_e = pd.merge(
    edf[edf["version"] == "generic"][["model", "scenario_id", "lang_pair", "drift"]],
    edf[edf["version"] == "localized"][["model", "scenario_id", "lang_pair", "drift"]],
    on=["model", "scenario_id", "lang_pair"], suffixes=("_generic", "_localized")
)
if len(pooled_e) >= 2:
    stat_pe, p_pe = stats.wilcoxon(pooled_e["drift_generic"], pooled_e["drift_localized"])
    d_pe = cohens_d(pooled_e["drift_localized"], pooled_e["drift_generic"])
    print(f"n={len(pooled_e)}  mean_drift_generic={pooled_e['drift_generic'].mean():.4f}  "
          f"mean_drift_localized={pooled_e['drift_localized'].mean():.4f}")
    print(f"Wilcoxon: W={stat_pe:.1f}, p={p_pe:.4f}, Cohen's d={d_pe:.3f}")
else:
    print("Not enough pooled paired records for this test")

## Table 3 — Model ranking by overall drift
*Added analysis layer.* Aggregates Table 2's per-dimension drift values into a single overall-drift score per model, generic condition, to give a ranked leaderboard.

In [ ]:
ranking_df = drift_df.groupby("model")[["drift_EN_TE", "drift_EN_TA", "drift_EN_KN"]].mean()
ranking_df["mean_drift_overall"] = ranking_df.mean(axis=1)
ranking_df = ranking_df.sort_values("mean_drift_overall", ascending=False).round(3)
print("TABLE 3: Model ranking by overall cross-lingual drift (generic condition, higher = more drift)")
print(ranking_df.to_string())

## Table 4 — Dimension ranking by overall drift
*Added analysis layer.* Same idea as Table 3, aggregated the other way — which Hofstede dimension shows the most cross-lingual drift, pooled across all models.

In [ ]:
dim_rank_df = drift_df.groupby("dimension")[["drift_EN_TE", "drift_EN_TA", "drift_EN_KN"]].mean()
dim_rank_df["mean_drift_overall"] = dim_rank_df.mean(axis=1)
dim_rank_df = dim_rank_df.sort_values("mean_drift_overall", ascending=False).round(3)
print("TABLE 4: Dimension ranking by overall cross-lingual drift (generic condition)")
print(dim_rank_df.to_string())

## Table 5 — Highest / lowest drift scenarios
*Added analysis layer.* Scenario-level (not just dimension-level) breakdown, pooled across models, generic condition — surfaces which individual scenarios drive the most / least drift, which the dimension-level tables above can hide.

In [ ]:
scenario_rows = []
for sid in jdf_generic["scenario_id"].unique():
    sub = jdf_generic[jdf_generic["scenario_id"] == sid]
    dim = sub["dimension"].iloc[0]
    mean_en = sub[sub["lang"] == "en"]["stance"].mean()
    for lang in DRAVIDIAN_LANGS:
        mean_l = sub[sub["lang"] == lang]["stance"].mean()
        if mean_en == mean_en and mean_l == mean_l:  # both not NaN
            scenario_rows.append({
                "scenario_id": sid, "dimension": dim, "lang": lang,
                "stance_EN": round(mean_en, 2), "stance_target": round(mean_l, 2),
                "drift": round(abs(mean_en - mean_l), 2),
            })
scenario_df = pd.DataFrame(scenario_rows)

print("Top 5 highest-drift (scenario, language) pairs:")
print(scenario_df.sort_values("drift", ascending=False).head(5).to_string(index=False))
print()
print("Top 5 lowest-drift (scenario, language) pairs:")
print(scenario_df.sort_values("drift", ascending=True).head(5).to_string(index=False))

## Table 6 — Script fidelity vs. drift correlation
*Added analysis layer.* Checks whether models that struggle to write the native script also tend to show larger (or smaller) value drift — a Spearman correlation across models between mean script fidelity and mean drift. Purely descriptive; with only as many data points as there are models, treat the p-value as indicative rather than confirmatory.

In [ ]:
fid_mean = fidelity_df.mean(axis=1)  # mean fidelity % across TE/TA/KN, per model
merged_corr = pd.DataFrame({
    "mean_fidelity": fid_mean,
    "mean_drift": ranking_df["mean_drift_overall"],
}).dropna()

print("Per-model fidelity vs. drift:")
print(merged_corr.round(3).to_string())

if len(merged_corr) >= 3:
    rho, p_rho = stats.spearmanr(merged_corr["mean_fidelity"], merged_corr["mean_drift"])
    print(f"\nSpearman correlation (fidelity vs. drift): rho={rho:.3f}, p={p_rho:.4f}, n={len(merged_corr)}")
else:
    print("\nToo few models with both fidelity and drift values for a correlation.")

## Table 7 — Agreement between judge-score drift and embedding drift
*Added analysis layer.* The pipeline produces two independent drift signals per (model, language): the LLM-judge's stance drift (Table 2) and LaBSE's semantic embedding drift (`edf`). This checks whether they agree — a sanity check on whether "value drift" as scored by the judge lines up with raw semantic drift in the response text.

In [ ]:
judge_by_model_lang = ranking_df[["drift_EN_TE", "drift_EN_TA", "drift_EN_KN"]].reset_index().melt(
    id_vars="model", var_name="lang_col", value_name="judge_drift"
)
judge_by_model_lang["lang_pair"] = judge_by_model_lang["lang_col"].str.replace("drift_EN_", "en→").str.lower()

embed_generic = edf[edf["version"] == "generic"].groupby(["model", "lang_pair"])["drift"].mean().reset_index()
embed_generic = embed_generic.rename(columns={"drift": "embed_drift"})

agreement_df = pd.merge(
    judge_by_model_lang[["model", "lang_pair", "judge_drift"]],
    embed_generic,
    on=["model", "lang_pair"], how="inner"
)
print("Judge drift vs. embedding drift, per model × language pair (generic condition):")
print(agreement_df.round(3).to_string(index=False))

if len(agreement_df) >= 3:
    rho2, p_rho2 = stats.spearmanr(agreement_df["judge_drift"], agreement_df["embed_drift"])
    print(f"\nSpearman correlation (judge drift vs. embedding drift): rho={rho2:.3f}, p={p_rho2:.4f}, n={len(agreement_df)}")
else:
    print("\nToo few matched (model, lang_pair) rows for a correlation.")

## Figure 1 — Drift heatmap by model × dimension (generic condition)

In [ ]:
for lang_code, col in [("te", "drift_EN_TE"), ("ta", "drift_EN_TA"), ("kn", "drift_EN_KN")]:
    pivot = drift_df.pivot_table(values=col, index="model", columns="dimension", aggfunc="mean")
    n_models = len(pivot)
    fig, ax = plt.subplots(figsize=(9, max(4, n_models * 0.65)))
    sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlOrRd",
                linewidths=0.5, ax=ax, vmin=0, vmax=2)
    ax.set_title(f"Cross-lingual Value Drift EN→{lang_code.upper()} by Model & Dimension (generic)", fontsize=12)
    ax.set_xlabel("Hofstede Dimension")
    ax.set_ylabel("Model")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"fig1_drift_heatmap_{lang_code}.pdf", bbox_inches="tight")
    plt.show()
    print(f"Figure 1 ({lang_code}) saved.")

## Figure 2 — EN vs TE vs TA vs KN stance per dimension (boxplot, generic)

In [ ]:
dims = jdf_generic["dimension"].unique()
fig, axes = plt.subplots(1, len(dims), figsize=(3.5 * len(dims), 4), sharey=True)
if len(dims) == 1:
    axes = [axes]

for ax, dim in zip(axes, dims):
    sub = jdf_generic[jdf_generic["dimension"] == dim]
    data = [sub[sub["lang"] == l]["stance"].values for l in LANGS]
    ax.boxplot(data, labels=[l.upper() for l in LANGS])
    ax.set_title(dim, fontsize=9)
    ax.set_ylim(0.5, 5.5)
    ax.set_ylabel("Stance (1=Western, 5=South Asian)" if ax is axes[0] else "")

plt.suptitle("Value Stance Distribution by Language and Dimension (generic condition)", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig2_stance_boxplot.pdf", bbox_inches="tight")
plt.show()
print("Figure 2 saved.")

## Figure 3 — Embedding drift EN→TE / EN→TA / EN→KN by model
Pooled across generic + localized, matching `03_embed_distance.py`'s own summary.

In [ ]:
embed_model_lang = edf.groupby(["model", "lang_pair"])["drift"].mean().reset_index()
target_pairs = [f"en→{l}" for l in DRAVIDIAN_LANGS]
embed_pivot = embed_model_lang[embed_model_lang["lang_pair"].isin(target_pairs)].pivot(
    index="model", columns="lang_pair", values="drift"
)
embed_pivot = embed_pivot[[c for c in target_pairs if c in embed_pivot.columns]]

fig, ax = plt.subplots(figsize=(8, 4))
embed_pivot.plot(kind="bar", ax=ax, color=["steelblue", "darkorange", "seagreen"], edgecolor="black")
ax.set_title("Semantic Embedding Drift by Model (LaBSE)", fontsize=12)
ax.set_ylabel("Mean Cosine Drift (1 - similarity)")
ax.set_xlabel("Model")
ax.legend(title="Language Pair")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig3_embed_drift.pdf", bbox_inches="tight")
plt.show()
print("Figure 3 saved.")

## Figure 4 — Model ranking by overall drift
*Added analysis layer.* Visualizes Table 3.

In [ ]:
fig, ax = plt.subplots(figsize=(8, max(3, len(ranking_df) * 0.5)))
ranking_df["mean_drift_overall"].sort_values().plot(kind="barh", ax=ax, color="indianred", edgecolor="black")
ax.set_xlabel("Mean Cross-lingual Drift (generic condition, pooled TE/TA/KN)")
ax.set_ylabel("Model")
ax.set_title("Model Ranking by Overall Cross-lingual Value Drift")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig4_model_ranking.pdf", bbox_inches="tight")
plt.show()
print("Figure 4 saved.")

## Figure 5 — Fidelity vs. drift scatter
*Added analysis layer.* Visualizes Table 6.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(merged_corr["mean_fidelity"], merged_corr["mean_drift"], s=80, edgecolor="black")
for model_name, row in merged_corr.iterrows():
    ax.annotate(model_name, (row["mean_fidelity"], row["mean_drift"]),
                textcoords="offset points", xytext=(5, 5), fontsize=8)
ax.set_xlabel("Mean Script Fidelity % (TE/TA/KN)")
ax.set_ylabel("Mean Cross-lingual Drift (generic condition)")
ax.set_title("Script Fidelity vs. Value Drift, by Model")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig5_fidelity_vs_drift.pdf", bbox_inches="tight")
plt.show()
print("Figure 5 saved.")

## Save final tables for the paper

In [ ]:
RES_DIR.mkdir(parents=True, exist_ok=True)

drift_df.to_csv(RES_DIR / "final_table.csv", index=False)
ranking_df.to_csv(RES_DIR / "model_ranking.csv")
dim_rank_df.to_csv(RES_DIR / "dimension_ranking.csv")
scenario_df.to_csv(RES_DIR / "scenario_level_drift.csv", index=False)
h3_judge_df.to_csv(RES_DIR / "h3_judge_generic_vs_localized.csv", index=False)
h3_embed_df.to_csv(RES_DIR / "h3_embed_generic_vs_localized.csv", index=False)
agreement_df.to_csv(RES_DIR / "judge_vs_embed_agreement.csv", index=False)

print("── MAIN RESULT: Model ranking by overall drift (generic condition) ──")
print(ranking_df.to_string())
print(f"\nAll results saved to {RES_DIR}/")